# 02-02 활성화 함수와 소프트맥스

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. 자주 쓰는 활성화 함수

In [ ]:
t = torch.linspace(-4, 4, 200)
acts = {"ReLU": nn.ReLU(), "GELU": nn.GELU(), "Sigmoid": nn.Sigmoid(), "Tanh": nn.Tanh()}
fig, axes = plt.subplots(1, 4, figsize=(16, 3.3))
for ax, (name, f) in zip(axes, acts.items()):
    ax.plot(t, f(t), color="tab:orange")
    ax.axhline(0, color="k", lw=0.8)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_ylim(-1.5, 4)
    ax.set_title(name)
plt.show()

In [ ]:
x = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])
for name, f in acts.items():
    print(f"{name:8s}", [round(v, 3) for v in f(x).tolist()])

## 2. 소프트맥스: 점수를 확률로

In [ ]:
logits = torch.tensor([2.0, 1.0, 0.1])  # 클래스 3개의 점수
exp = torch.exp(logits)
probs = exp / exp.sum()
print("직접 계산:", probs)
print("F.softmax:", F.softmax(logits, dim=-1))
print("합:", probs.sum().item())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
labels = ["cat", "dog", "bird"]
axes[0].bar(labels, logits, color="tab:gray")
axes[0].set_title("logits")
axes[1].bar(labels, probs, color="tab:orange")
axes[1].set_title("softmax(logits)")
axes[1].set_ylim(0, 1)
plt.show()

In [ ]:
batch_logits = torch.randn(4, 3)  # (batch=4, 클래스=3)
p = F.softmax(batch_logits, dim=-1)  # (4, 3), 마지막 축을 따라 합이 1
print(p.sum(dim=-1))

## 3. 교차 엔트로피 손실

In [ ]:
target = torch.tensor(0)  # 정답은 0번 클래스(cat)
manual = -torch.log(probs[target])
builtin = F.cross_entropy(logits.unsqueeze(0), target.unsqueeze(0))  # (1, 3), (1,)
print(manual.item(), builtin.item())

In [ ]:
logits_b = torch.randn(4, 3)  # (batch=4, 클래스=3)
targets_b = torch.tensor([0, 2, 1, 2])  # (4,)
loss_fn = nn.CrossEntropyLoss()
print(loss_fn(logits_b, targets_b))

In [ ]:
p = torch.linspace(0.01, 1, 100)
plt.figure(figsize=(4, 3))
plt.plot(p, -torch.log(p), color="tab:red")
plt.xlabel("probability of correct class")
plt.ylabel("loss = -log(p)")
plt.show()